# <span style="color:purple"> Find and Remove Instances of Pressure Fluctuations Due to Platelet Ice Formation </span>

<span style="color:purple"> Craven et al. (2014) reports sporadic pressure decreases believed to be associated with frazil accretion and platelet ice formation, which increases the buoyancy of the moorings. Whether transient or permanent (where the uppermost instrument has become lodged in the base of the ice shelf (Craven et al., 2014)), such pressure fluctuations can affect the longer-term consistency of data recordings. </span>

## <span style="color:purple"> Import Data for Analysis </span>

In [2]:
import pandas as pd

df = pd.read_parquet("../data/proc/03_filtered.parquet")

## <span style="color:purple"> Locate Fluctuations </span>

### <span style="color:purple"> Quantitatively locate pressure changes </span>

In [10]:
import numpy as np

# Define a function to detect pressure change events
def detect_pressure_events(df, boreholes, threshold, window_days):
    """
    boreholes:          since we only care about AM01, AM04, and AM05 (where marine ice accretion occurs), we specify these boreholes here
    threshold:          minimum pressure change in dbar to consider an event
    window_days:        time window to look for pressure changes (in days)
    """
    # Create an empty list to store pressure events that have been detected for this threshold and window of days
    results = []

    # Filter the DataFrame by only the boreholes that we care about
    df_sub = df[df["borehole"].isin(boreholes)].copy()
    # Ensure that the DataFrame is sorted by time for rolling calculations
    df_sub = df_sub.sort_values(["borehole", "cat", "time"])

    # Iterate over each borehole and cat combination
    for (bh, cat), group in df_sub.groupby(["borehole", "cat"]):
        # Check if each pressure value is NaN
        # If all are NaN skip this MicroCAT (i.e. we can't infer pressure changes if there are no pressure measurements)
        if group["pressure"].isna().all():
            continue

        # Make time the index of the group
        group = group.set_index("time")
        # For each timestamp, calculate the rolling minimum and maximum pressure over the specified window of days
        # .rolling() is a pandas method that creates a rolling window of specified size
        # .rolling() itself iterates over the time index 
        p_min = group["pressure"].rolling(f"{window_days}D").min()
        p_max = group["pressure"].rolling(f"{window_days}D").max()
        # Calculate the range of pressure values in the rolling 10-day window
        p_range = p_max - p_min

        # Filters down to timestamps where the pressure range exceeds the specified threshold
        exceedances = p_range[p_range > threshold]

        # If there are exceedances for this cat, we want to cluster them into distinct events
        if not exceedances.empty:
            # Before, we set time as a pandas Index, but this is different from a pandas Series
            # So we convert it back to a Series to use .diff() method
            times = exceedances.index.to_series()
            
            # Calculate the time difference between consecutive exceedances
            # If the time difference is greater than the specified window of days, consider it a separate event
            # gaps=True where a new event starts
            gaps = times.diff() > pd.Timedelta(days=window_days)
            
            # Assign an event ID to each event, where the event ID increments by 1 each time a new event starts (True in gaps)
            # E.g. if gaps is [False, False, True, False, True, False], then event_id will be [0, 0, 1, 1, 2, 2]
            event_id = gaps.cumsum()
            
            # Now we iterate over each distinct event ID
            for eid, event_group in exceedances.groupby(event_id):
                # Record the first and last timestamps associated with the event
                event_start = event_group.index[0] - pd.Timedelta(days=5)
                event_end = event_group.index[-1]
                
                # Extract the pressure values for this event
                event_pressure = group.loc[event_start:event_end, "pressure"]
                
                # Denote the pressure at the start of the event as the reference point for calculating pressure changes
                pressure_start = event_pressure.iloc[0]
                # Calculate the cumulative pressure change from the start of the event
                pressure_cumulative = (event_pressure - pressure_start).abs()

                # Define the total range of pressure change during the event
                total_range = pressure_cumulative.max()

                # Create a boolean mask to identify timestamps where the cumulative pressure change exceeds 10% of the total range during the event
                transition_mask = pressure_cumulative >= 0.02 * total_range
                # Define the transition start time as when the pressure has deviated by more than 10%
                transition_start = pressure_cumulative[transition_mask].index[0]

                # Calculate the rate of negative pressure change
                pressure_rate = event_pressure.diff()
                # Smooth it slightly to avoid noise triggering a false "settled"
                pressure_rate_smooth = pressure_rate.rolling("2h").mean()
                # The transition is where the rate of change is high - find the peak rate of negative change during the event
                peak_rate = pressure_rate_smooth.min()
                # Create a boolean mask to identify timestamps where the rate of pressure change exceeds 10% of the peak rate during the event
                active_mask = pressure_rate_smooth <= 0.2 * peak_rate

                # Define the transitions end at last rapid change
                transition_end = pressure_rate_smooth[active_mask].index[-1]

                # Find the presure value at the transition start and end times
                pressure_before = event_pressure.loc[:transition_start].iloc[0]
                pressure_after = event_pressure.loc[transition_end:].iloc[0]
                # Find the pressure range for the actual transition period
                transition_range = abs(pressure_after - pressure_before)

                # Check if the pressure reverts to within 0.5 dbar of the starting pressure
                post_event = group.loc[transition_end:, "pressure"]
                # Create a boolean mask to identify timestamps where the pressure has reverted to within 0.5 dbar
                reverted = post_event[abs(post_event - pressure_start) < 0.5]

                if not reverted.empty:
                    duration = (reverted.index[0] - transition_start).total_seconds() / (3600 * 24)  # in days
                else:
                    duration = np.nan  # if it never reverts, we can set duration to NaN or some other value to indicate this

                # For each event, record the relevant information
                results.append({
                    "borehole": bh,
                    "cat": cat,
                    "event_start": event_start,
                    "event_end": event_end,
                    "transition_start": transition_start,
                    "transition_end": transition_end,
                    "transition_time": (transition_end - transition_start).total_seconds() / (3600 * 24),  # in days
                    "duration": duration,
                    "pressure_range": transition_range,
                    "reverts": not reverted.empty
                })

    # Since results was a list of dictionaries, we now convert it to a DataFrame
    events_df = pd.DataFrame(results)

    # Now group across cats to find which cats were affected by the same event
    # Events at the same borehole within window_days of each other are likely the same event
    
    # Create a new empty list to store the final output, where events will be grouped across cats
    summary = []

    # Iterate over each borehole and the events recorded for that borehole
    for bh, bh_events in events_df.groupby("borehole"):
        # Sort events chronologically using event start time 
        bh_events = bh_events.sort_values("event_start")

        # Initialise cluster ID, to represent distinct events as this borehole
        cluster_id = 0
        # Initialise an empty list to store assigned cluster number for each event
        clusters = []
        # Initialise last_start to the earliest possible timestamp, so that the first event will always start a new cluster
        last_start = pd.Timestamp("2000-01-01")

        # Iterate over each event for this borehole
        for _, row in bh_events.iterrows():
            # If this event started more than 10 days after the previous one, it is a new event, so increment the cluster ID
            if (row["event_start"] - last_start) > pd.Timedelta(days=window_days):
                cluster_id += 1
            
            # Assign the current cluster ID to the event
            clusters.append(cluster_id)
           
            # Update last_start to the start time of the current event for the next iteration
            last_start = row["event_start"]

        # Create a copy to avoid a pandas warning about modifying a slice
        bh_events = bh_events.copy()
        # Create a new column in the DataFrame to store the assigned cluster ID for each event
        bh_events["cluster"] = clusters

        # Iterate over each cluster of events for this borehole
        for cid, cluster in bh_events.groupby("cluster"):
            # For each cluster, record the relevent information in the summary list
            summary.append({
                "borehole": bh,
                "transition_start": cluster["transition_start"].min(),
                "transition_end": cluster["transition_end"].max(),
                "transition_time": cluster["transition_time"].max(),
                "duration": cluster["duration"].max(),
                "pressure_range": cluster["pressure_range"].max(),
                "reverts": cluster["reverts"].any(),
                "cats_affected": sorted(cluster["cat"].unique()),
            })

    # Since summary was a list of dictionaries, we now convert it to a DataFrame and output it from the function
    return pd.DataFrame(summary)


events = detect_pressure_events(df, ["AM01", "AM04", "AM05"], 10, 50)
print(events.to_string(index=False))

borehole    transition_start      transition_end  transition_time  duration  pressure_range  reverts      cats_affected
    AM04 2008-06-26 01:28:26 2008-06-27 05:59:11         1.187488 11.437454          15.295     True [1972, 1973, 1974]
    AM04 2009-07-06 11:58:42 2009-07-11 15:00:01         5.125000       NaN          13.875    False [1972, 1973, 1974]
    AM04 2010-08-19 17:30:02 2010-08-24 17:00:01         4.979155       NaN          12.905    False       [1973, 1974]
    AM05 2011-08-04 01:00:01 2012-01-14 01:30:01       163.020833       NaN          27.022    False       [7322, 7326]
    AM05 2012-05-28 14:00:01 2012-07-20 12:30:01        52.937500       NaN          15.933    False       [7322, 7326]


### <span style="color:purple"> Save Dataframe as Parquet </span>

In [8]:
# Save to parquet file for efficient storage and future use
# Note: parquet is a columnar storage format that is more efficient for large datasets and preserves data types better than CSV
# Note: parquet files can be read back into pandas with pd.read_parquet("data/proc/pressure_changes.parquet")
# Note: this will overwrite any existing file at this location
events.to_parquet("../data/proc/pressure_changes.parquet", index=False)